# 2. Embeddings and Clustering

**AI and Economics Summer School** — Andrea Ciccarone

We take a corpus of real documents, turn each one into a vector with a pretrained
embedding model, and then find structure in those vectors without using any labels.

The embedding model runs locally and is free. No API key is needed anywhere in
this notebook.

In [ ]:
!pip -q install sentence-transformers

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_20newsgroups
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sentence_transformers import SentenceTransformer

plt.rcParams["figure.dpi"] = 120
rng = np.random.default_rng(0)

## 1. A corpus

Four newsgroups with clearly different subject matter. We keep the true labels
aside so that at the end we can check whether the unsupervised method found
something real. **In your own research you will not have this luxury**, which is
exactly why the validation habits below matter.

In [ ]:
data = fetch_20newsgroups(
    subset="train",
    categories=["rec.sport.baseball", "sci.med", "talk.politics.guns", "comp.graphics"],
    remove=("headers", "footers", "quotes"), random_state=0)

# NOTE: fetch_20newsgroups returns categories in sorted order, not the order you
# asked for. Always read the names off the object.
names = data.target_names
print("label order:", names)

docs = [d.strip() for d in data.data]
labels = np.array(data.target)
keep = [i for i, d in enumerate(docs)
        if 200 < len(d) < 3000 and max((len(t) for t in d.split()), default=0) < 40]
docs = [docs[i] for i in keep][:800]
labels = labels[keep][:800]

print(len(docs), "documents")
print("\n--- example ---\n")
print(docs[0][:400])

## 2. Embed

One line. The model downloads once (about 90 MB) and then runs on CPU.

`all-MiniLM-L6-v2` is small and fast. It is a reasonable default for exploratory
work; for a paper you would compare against a larger model and report which one
you used, with its version.

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")
E = model.encode(docs, batch_size=64, show_progress_bar=True,
                 normalize_embeddings=True)   # normalize -> cosine similarity = dot product
print("embedding matrix:", E.shape)

Note `normalize_embeddings=True`. Cosine similarity is the standard similarity
measure for embeddings, and normalising once up front means every dot product
below is already a cosine.

In [ ]:
# Similarity between two documents from the same group vs. different groups.
same = np.where(labels == labels[0])[0][1]
diff = np.where(labels != labels[0])[0][0]

print("doc 0 is:", names[labels[0]])
print(f"cosine with another {names[labels[same]]:22s}: {E[0] @ E[same]:.3f}")
print(f"cosine with a       {names[labels[diff]]:22s}: {E[0] @ E[diff]:.3f}")

> Absolute cosine values are not interpretable on their own. A cosine of 0.31 is
> not "31% similar". **Differences and rankings** are what carry meaning.

## 3. Look at it: PCA

PCA first, because it is linear, deterministic, and distances survive it.

In [ ]:
pca = PCA(n_components=2, random_state=0)
Z = pca.fit_transform(E)

fig, ax = plt.subplots(figsize=(6, 4.5))
for k, name in enumerate(names):
    m = labels == k
    ax.scatter(Z[m, 0], Z[m, 1], s=10, alpha=0.6, label=name)
ax.legend(frameon=False, fontsize=8)
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
ax.set_title("Newsgroup embeddings, coloured by true label")
plt.tight_layout(); plt.show()

The colours separate even though PCA never saw the labels. That is the whole
promise of embeddings: the geometry already encodes the meaning.

Two components explain only a small share of the variance, so do not over-read
this picture. Cluster on the full embedding, not on these two dimensions.

## 4. Choose k

Elbow and silhouette give you a range, not an answer.

In [ ]:
ks = range(2, 11)
inertia, sil = [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(E)
    inertia.append(km.inertia_)
    sil.append(silhouette_score(E, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
axes[0].plot(list(ks), inertia, "o-"); axes[0].set_title("Elbow"); axes[0].set_xlabel("k")
axes[1].plot(list(ks), sil, "o-", color="#378C91"); axes[1].set_title("Silhouette"); axes[1].set_xlabel("k")
plt.tight_layout(); plt.show()

## 5. Cluster, then *read the clusters*

This is the step people skip, and it is the one that decides whether your
unsupervised result is a finding or an artefact.

For each cluster we pull the documents closest to the centroid and read them.

In [ ]:
K = 4
km = KMeans(n_clusters=K, n_init=10, random_state=0).fit(E)

for c in range(K):
    idx = np.where(km.labels_ == c)[0]
    d = E[idx] @ km.cluster_centers_[c]          # similarity to own centroid
    closest = idx[np.argsort(-d)[:3]]
    print("=" * 70)
    print(f"CLUSTER {c}  (n = {len(idx)})")
    for i in closest:
        print(f"  [true label: {names[labels[i]]}]")
        print("   ", docs[i][:180].replace("\n", " "), "...")
    print()

Now name each cluster from what you just read, *before* looking at the next cell.

In [ ]:
# Only now do we check against the held-out truth.
import pandas as pd
ct = pd.crosstab(pd.Series(km.labels_, name="cluster"),
                 pd.Series([names[l] for l in labels], name="true"))
display(ct)
print("Adjusted Rand index:", round(adjusted_rand_score(labels, km.labels_), 3))

## 6. Stability

A clustering you cannot reproduce is not a result. Re-run with different seeds and
on a random half of the corpus, and see whether the same structure appears.

In [ ]:
for seed in [0, 1, 2, 3]:
    lab = KMeans(n_clusters=K, n_init=10, random_state=seed).fit_predict(E)
    print(f"seed {seed}: ARI vs truth = {adjusted_rand_score(labels, lab):.3f}")

half = rng.choice(len(E), size=len(E) // 2, replace=False)
lab_half = KMeans(n_clusters=K, n_init=10, random_state=0).fit_predict(E[half])
print("half sample: ARI vs truth =", round(adjusted_rand_score(labels[half], lab_half), 3))

## Exercises

1. Re-run the whole notebook with `K = 6` and `K = 3`. At which values can you still
   name every cluster from reading its documents? That is usually the binding constraint.
2. Swap `all-MiniLM-L6-v2` for `all-mpnet-base-v2` (larger, slower). Does the
   cluster structure change? Would your substantive conclusion change?
3. Add a fifth newsgroup that is close in subject matter to an existing one
   (e.g. `sci.space` alongside `sci.med`). Does k-means still separate them?
4. Compare with a bag-of-words baseline: `TfidfVectorizer` followed by
   `TruncatedSVD(100)`, then the same k-means. How much do embeddings actually buy you here?